# 02 — Nettoyage : une ligne par vente de logement

Objectif : passer des lignes brutes de DVF à un tableau propre où **chaque ligne est la vente
d'un seul logement** (maison ou appartement), avec son prix, ses caractéristiques et sa position.

Le résultat est sauvegardé dans `data/processed/logements_paca.parquet`.

In [9]:
import pandas as pd
from pathlib import Path

# Remonter les dossiers jusqu'à trouver data/raw (fonctionne où que soit le notebook)
racine = Path.cwd()
while not (racine / "data" / "raw").exists() and racine != racine.parent:
    racine = racine.parent

fichiers = sorted((racine / "data" / "raw").glob("*.csv.gz"))
print(f"{len(fichiers)} fichiers trouvés dans {racine / 'data' / 'raw'}")

# Les codes (postal, commune, département, parcelle) sont des identifiants, pas des nombres :
# on les lit comme du texte pour garder les zéros de tête ("04" et non 4)
types_texte = {"code_postal": str, "code_commune": str, "code_departement": str, "id_parcelle": str}
df = pd.concat(
    (pd.read_csv(f, low_memory=False, dtype=types_texte) for f in fichiers),
    ignore_index=True,
)
print(f"Lignes brutes : {len(df):,}")

entonnoir = {"Lignes brutes": len(df)}

30 fichiers trouvés dans c:\Users\salwa\estimateur-immo-dvf\data\raw
Lignes brutes : 1,720,509


## 1. Ne garder que les ventes classiques

In [10]:
# On écarte les échanges, adjudications, ventes en l'état futur d'achèvement (VEFA), etc.
ventes = df[df["nature_mutation"] == "Vente"].copy()
entonnoir["Lignes de type « Vente »"] = len(ventes)
entonnoir["Ventes uniques"] = ventes["id_mutation"].nunique()
print(f"Lignes 'Vente' : {len(ventes):,}")
print(f"Ventes uniques : {ventes['id_mutation'].nunique():,}")

Lignes 'Vente' : 1,576,006
Ventes uniques : 630,880


## 2. Identifier les biens distincts de chaque vente (piège n°2)

In [11]:
# Les lignes sans type_local sont du terrain nu : on les met de côté pour l'instant.
# Un même bien peut être répété (une ligne par nature de culture du terrain) : on dédoublonne.
locaux = ventes.dropna(subset=["type_local"]).drop_duplicates(
    subset=["id_mutation", "id_parcelle", "type_local",
            "surface_reelle_bati", "nombre_pieces_principales", "lot1_numero"]
)

# Pour chaque vente : combien de maisons, d'appartements, de dépendances, d'autres locaux ?
comptes = locaux.groupby("id_mutation")["type_local"].value_counts().unstack(fill_value=0)
comptes.head()

type_local,Appartement,Dépendance,Local industriel. commercial ou assimilé,Maison
id_mutation,,,,
2021-1397162,0,0,0,1
2021-1397163,0,1,0,1
2021-1397164,0,0,0,1
2021-1397165,0,0,0,1
2021-1397166,2,1,0,0


## 3. Garder les ventes d'un seul logement (piège n°1)

In [12]:
nb_logements = comptes.get("Maison", 0) + comptes.get("Appartement", 0)
nb_dependances = comptes.get("Dépendance", 0)
nb_autres = comptes.sum(axis=1) - nb_logements - nb_dependances  # locaux commerciaux, etc.

ventes_simples = comptes.index[(nb_logements == 1) & (nb_autres == 0)]
entonnoir["Ventes avec au moins un bâtiment"] = len(comptes)
entonnoir["Ventes d'un seul logement"] = len(ventes_simples)
print(f"Ventes d'un seul logement : {len(ventes_simples):,} "
      f"sur {len(comptes):,} ventes avec au moins un bâtiment")

Ventes d'un seul logement : 450,363 sur 560,698 ventes avec au moins un bâtiment


## 4. Construire le tableau final : une ligne par vente

In [13]:
colonnes = ["id_mutation", "date_mutation", "valeur_fonciere", "type_local",
            "surface_reelle_bati", "nombre_pieces_principales",
            "code_postal", "code_commune", "nom_commune", "code_departement",
            "longitude", "latitude"]

logements = locaux[
    locaux["type_local"].isin(["Maison", "Appartement"])
    & locaux["id_mutation"].isin(ventes_simples)
][colonnes].set_index("id_mutation")

# Nombre de dépendances (caves, garages...) : ce sera une variable du modèle
logements["nb_dependances"] = nb_dependances.reindex(logements.index).fillna(0).astype(int)

# Surface de terrain totale de la vente, sans compter deux fois la même parcelle
terrain = (
    ventes.drop_duplicates(subset=["id_mutation", "id_parcelle", "code_nature_culture", "surface_terrain"])
    .groupby("id_mutation")["surface_terrain"].sum()
)
logements["surface_terrain"] = terrain.reindex(logements.index).fillna(0)
# Pour un appartement, le terrain est celui de toute la copropriété : sans intérêt
logements.loc[logements["type_local"] == "Appartement", "surface_terrain"] = 0

logements["date_mutation"] = pd.to_datetime(logements["date_mutation"])

print(f"Une ligne par vente ? {logements.index.is_unique}")
print(f"Logements : {len(logements):,}")
logements.head()

Une ligne par vente ? True
Logements : 450,363


,date_mutation,valeur_fonciere,type_local,surface_reelle_bati,nombre_pieces_principales,code_postal,code_commune,nom_commune,code_departement,longitude,latitude,nb_dependances,surface_terrain
id_mutation,,,,,,,,,,,,,
2021-40424,2021-01-05,81960.0,Appartement,28.0,1.0,04800,04094,Gréoux-les-Bains,04,5.894395,43.759373,0,0.0
2021-40426,2021-01-05,405500.0,Maison,86.0,3.0,04400,04226,Uvernet-Fours,04,6.610453,44.359152,0,401.0
2021-40428,2021-01-07,20000.0,Maison,65.0,2.0,04380,04217,Thoard,04,6.147419,44.149455,0,29.0
2021-40429,2021-01-07,254000.0,Appartement,68.0,4.0,04260,04006,Allos,04,6.623223,44.243794,2,0.0
2021-40430,2021-01-05,233000.0,Maison,112.0,3.0,05110,04066,Curbans,04,6.039241,44.429802,0,592.0


## 5. Supprimer les valeurs absurdes

In [14]:
avant = len(logements)

logements = logements.dropna(subset=["valeur_fonciere", "surface_reelle_bati", "longitude", "latitude"])
logements = logements[
    (logements["valeur_fonciere"] >= 10_000)          # ventes symboliques
    & (logements["surface_reelle_bati"] >= 9)         # erreurs de saisie
    & (logements["nombre_pieces_principales"] >= 1)
]

# Prix au m² : on retire les 1 % les plus extrêmes de chaque côté, par type de bien
logements["prix_m2"] = logements["valeur_fonciere"] / logements["surface_reelle_bati"]
bas = logements.groupby("type_local")["prix_m2"].transform(lambda s: s.quantile(0.01))
haut = logements.groupby("type_local")["prix_m2"].transform(lambda s: s.quantile(0.99))
logements = logements[(logements["prix_m2"] >= bas) & (logements["prix_m2"] <= haut)]

entonnoir["Après suppression des valeurs aberrantes"] = len(logements)
print(f"Supprimés : {avant - len(logements):,} ({(avant - len(logements)) / avant:.1%})")
print(f"Restants : {len(logements):,}")
logements.groupby("type_local")["prix_m2"].describe()

Supprimés : 16,274 (3.6%)
Restants : 434,089


,count,mean,std,min,25%,50%,75%,max
type_local,,,,,,,,
Appartement,290404.0,3946.638402,1846.864915,724.637681,2575.757576,3680.000000,4991.995727,11515.151515
Maison,143685.0,4304.390309,2365.259526,666.666667,2780.000000,3823.529412,5208.333333,17571.428571


**Pourquoi ces suppressions.** On retire 16 274 ventes, soit 3,6 % des logements :

- les prix inférieurs à 10 000 € correspondent à des cessions symboliques ou familiales, pas au prix du marché ;
- les surfaces de moins de 9 m² ou les logements sans pièce principale sont des erreurs de saisie ;
- les 1 % de prix au m² les plus bas et les plus hauts, par type de bien, contiennent surtout des erreurs (prix saisi pour une autre surface, bien en ruine, vente partielle…).

**Limite assumée.** Le seuil haut coupe les appartements au-delà d'environ 11 500 €/m² et les maisons au-delà d'environ 17 500 €/m². Or une partie de ces ventes est bien réelle : à Saint-Tropez, Ramatuelle ou Saint-Jean-Cap-Ferrat, ces prix existent. **Le modèle ne couvrira donc pas l'immobilier de grand luxe**, ce qui est cohérent avec l'objectif d'estimer des biens « courants ».

Autre limite : les ventes en l'état futur d'achèvement (VEFA, 124 490 lignes) ont été écartées dès l'étape 1. Le modèle estime donc des logements **anciens**, pas du neuf.

## 6. Résumé : l'entonnoir de nettoyage

In [15]:
pd.DataFrame({"Nombre": entonnoir}).style.format("{:,.0f}")

,Nombre
Lignes brutes,"1,720,509"
Lignes de type « Vente »,"1,576,006"
Ventes uniques,"630,880"
Ventes avec au moins un bâtiment,"560,698"
Ventes d'un seul logement,"450,363"
Après suppression des valeurs aberrantes,"434,089"


## 7. Sauvegarde

In [16]:
# Vérification : les codes doivent garder leurs zéros de tête ("04", "04800")
assert logements["code_departement"].str.len().eq(2).all(), "Codes département mal lus"

dossier_sortie = racine / "data" / "processed"
dossier_sortie.mkdir(exist_ok=True)
logements.to_parquet(dossier_sortie / "logements_paca.parquet")
print(f"Sauvegardé : {dossier_sortie / 'logements_paca.parquet'}")

Sauvegardé : c:\Users\salwa\estimateur-immo-dvf\data\processed\logements_paca.parquet
